In [30]:
import numpy as np
import pandas as pd
import csv
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

user_info = pd.read_csv("data/A榜-训练集_分布式光伏发电预测_基本信息.csv",encoding = 'gbk')

user_info.head()

,光伏用户名称,光伏用户编号,装机容量(kW),经度,纬度
0,f1光伏发电用户,f1,239.220,119.218560,26.042931
1,f2光伏发电用户,f2,396.000,118.124457,24.695315
2,f3光伏发电用户,f3,397.870,117.002056,25.112496
3,f4光伏发电用户,f4,332.395,117.854904,26.744673
4,f5光伏发电用户,f5,201.140,120.022313,26.872516


In [31]:
weather_data = pd.read_csv("data/A榜-训练集_分布式光伏发电预测_气象变量数据.csv",encoding = 'gbk')
data = weather_data[weather_data['光伏用户编号'] == 'f1']
data

,光伏用户编号,时间,气压(Pa）,相对湿度（%）,云量,10米风速（10m/s）,10米风向（°),温度（K）,辐照强度（J/m2）,降水（m）,100m风速（100m/s）,100m风向（°)
0,f1,2022-1-3 0:00,100361.6094,79.3513,0.007812,1.16950,270.0,280.6320,0.0,0.000229,0.00000,270.0
1,f1,2022-1-3 0:15,100351.1476,76.8741,0.007005,1.14900,270.0,280.6308,0.0,0.000229,0.00000,270.0
2,f1,2022-1-3 0:30,100341.8609,74.9771,0.006890,1.15560,270.0,280.5719,0.0,0.000229,0.00000,270.0
3,f1,2022-1-3 0:45,100333.2668,73.6136,0.007236,1.18070,270.0,280.4776,0.0,0.000229,0.00000,270.0
4,f1,2022-1-3 1:00,100324.8828,72.7366,0.007812,1.21570,270.0,280.3705,0.0,0.000229,0.00000,270.0
...,...,...,...,...,...,...,...,...,...,...,...,...
46363,f1,2023-4-30 22:45,99475.2637,62.0537,0.378860,0.98215,270.0,286.9740,0.0,0.023335,0.45378,270.0
46364,f1,2023-4-30 23:00,99467.6875,63.9828,0.445310,1.03310,270.0,286.8915,0.0,0.021881,0.47551,270.0
46365,f1,2023-4-30 23:15,99463.2729,69.3050,0.555090,0.96405,270.0,287.0081,0.0,0.018652,0.35761,270.0
46366,f1,2023-4-30 23:30,99460.7843,76.6493,0.686570,0.81829,270.0,287.2474,0.0,0.014395,0.15814,270.0


In [32]:
from datetime import datetime, timedelta
power_data = pd.read_csv("data\A榜-训练集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')

df = pd.DataFrame(power_data)

# 创建新的DataFrame,将p1到p96作为行索引
new_data = []
start_date = datetime(2022, 1, 3, 0, 15)  # 起始时间为2022-01-03 00:15
for i in range(df.shape[0]):
    row = df.iloc[i]
    for j in range(1, 97):
        time = start_date + timedelta(minutes=15*(j-1))
        new_data.append([row['光伏用户编号'], row['综合倍率'], time])

power_data = pd.DataFrame(new_data, columns=['光伏用户编号', '综合倍率', '时间'])
power_data

,光伏用户编号,综合倍率,时间
0,f1,80,2022-01-03 00:15:00
1,f1,80,2022-01-03 00:30:00
2,f1,80,2022-01-03 00:45:00
3,f1,80,2022-01-03 01:00:00
4,f1,80,2022-01-03 01:15:00
...,...,...,...
416347,f9,8000,2022-01-03 23:00:00
416348,f9,8000,2022-01-03 23:15:00
416349,f9,8000,2022-01-03 23:30:00
416350,f9,8000,2022-01-03 23:45:00


In [33]:
user_info_test = pd.read_csv("data/A榜-测试集_分布式光伏发电预测_基本信息.csv",encoding = 'gbk')
weather_test = pd.read_csv("data/A榜-测试集_分布式光伏发电预测_气象变量数据.csv",encoding = 'gbk')
power_test = pd.read_csv("data\A榜-测试集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')

In [34]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Convert '时间' column to datetime64 data type
weather_data['时间'] = pd.to_datetime(weather_data['时间'])
power_data['时间'] = pd.to_datetime(power_data['时间'])

# Now merge the DataFrames
train_data = pd.merge(power_data, weather_data, on=['光伏用户编号', '时间'],how='left')
train_data = pd.merge(train_data, user_info, on='光伏用户编号')

train_data

,光伏用户编号,综合倍率,时间,气压(Pa）,相对湿度（%）,云量,10米风速（10m/s）,10米风向（°),温度（K）,辐照强度（J/m2）,降水（m）,100m风速（100m/s）,100m风向（°),光伏用户名称,装机容量(kW),经度,纬度
0,f1,80,2022-01-03 00:15:00,100351.1476,76.8741,0.007005,1.14900,270.0,280.6308,0.0,2.288800e-04,0.000000,270.0,f1光伏发电用户,239.22,119.218560,26.042931
1,f1,80,2022-01-03 00:30:00,100341.8609,74.9771,0.006890,1.15560,270.0,280.5719,0.0,2.288800e-04,0.000000,270.0,f1光伏发电用户,239.22,119.218560,26.042931
2,f1,80,2022-01-03 00:45:00,100333.2668,73.6136,0.007236,1.18070,270.0,280.4776,0.0,2.288800e-04,0.000000,270.0,f1光伏发电用户,239.22,119.218560,26.042931
3,f1,80,2022-01-03 01:00:00,100324.8828,72.7366,0.007812,1.21570,270.0,280.3705,0.0,2.288800e-04,0.000000,270.0,f1光伏发电用户,239.22,119.218560,26.042931
4,f1,80,2022-01-03 01:15:00,100316.2263,72.2992,0.008389,1.25200,270.0,280.2730,0.0,2.288800e-04,0.071884,270.0,f1光伏发电用户,239.22,119.218560,26.042931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416347,f9,8000,2022-01-03 23:00:00,101926.1875,65.1822,0.000000,0.80490,270.0,285.0117,0.0,0.000000e+00,0.000000,270.0,f9光伏发电用户,6000.00,117.740547,24.077638
416348,f9,8000,2022-01-03 23:15:00,101922.3264,64.1051,0.000000,0.77014,270.0,284.9275,0.0,2.410000e-20,0.000000,270.0,f9光伏发电用户,6000.00,117.740547,24.077638
416349,f9,8000,2022-01-03 23:30:00,101916.7581,62.8999,0.000000,0.71538,270.0,284.8174,0.0,4.520000e-20,0.000000,270.0,f9光伏发电用户,6000.00,117.740547,24.077638
416350,f9,8000,2022-01-03 23:45:00,101910.2170,61.7351,0.000000,0.65664,270.0,284.6963,0.0,4.370000e-20,0.000000,270.0,f9光伏发电用户,6000.00,117.740547,24.077638


In [35]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
# 对 f1-f9 进行独热编码
encoder = OneHotEncoder(sparse=False)
feature1_values = train_data['光伏用户编号'].values.reshape(-1, 1)
feature1_encoded = encoder.fit_transform(feature1_values)
feature1_encoded

D:\anaconda1\envs\pm3env\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]])

In [36]:
from sklearn.preprocessing import MinMaxScaler
n_tasks = 96
# 特征工程
train_data['时间'] = pd.to_datetime(train_data['时间'])
train_data['月份'] = train_data['时间'].dt.month
train_data['日期'] = train_data['时间'].dt.day
train_data['小时'] = train_data['时间'].dt.hour
train_data['分钟'] = train_data['时间'].dt.minute

# 按时间排序数据
train_data = train_data.sort_values(by='时间')

# 选择特征和目标变量
X_train = train_data.drop('光伏用户编号', axis=1)
X_train = X_train.drop('时间', axis=1)
X_train = X_train.drop('光伏用户名称', axis=1)
X_train = np.column_stack((X_train, feature1_encoded))

# 标准化或归一化特征
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

n_features = X_train.shape[1]
X_train

array([[0.        , 0.72612204, 0.62103422, ..., 0.        , 0.        ,
        0.        ],
       [0.00505051, 0.90621506, 0.67375769, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.26627397, 0.96838111, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.00505051, 0.043532  , 0.52034637, ..., 0.        , 0.        ,
        1.        ],
       [0.00505051, 0.043532  , 0.52034637, ..., 0.        , 0.        ,
        1.        ],
       [1.        , 0.93636779, 0.35515705, ..., 0.        , 0.        ,
        1.        ]])

In [37]:
X_train = np.ravel(X_train)
len(X_train)
# X_train 转换为(n_samples, n_tasks, n_features)
n_tasks = 96  # 假设有2个任务
n_features = 27

# 将 X_train 重塑为 (n_samples, n_tasks, n_features) 的形状
X_train = X_train.reshape(-1, n_tasks, n_features)
print(X_train.shape)

(4337, 96, 27)


In [38]:
power_data_or = pd.read_csv("data\A榜-训练集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')
y_train = power_data_or[['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27', 'p28', 'p29', 'p30', 'p31', 'p32', 'p33', 'p34', 'p35', 'p36', 'p37', 'p38', 'p39', 'p40', 'p41', 'p42', 'p43', 'p44', 'p45', 'p46', 'p47', 'p48', 'p49', 'p50', 'p51', 'p52', 'p53', 'p54', 'p55', 'p56', 'p57', 'p58', 'p59', 'p60', 'p61', 'p62', 'p63', 'p64', 'p65', 'p66', 'p67', 'p68', 'p69', 'p70', 'p71', 'p72', 'p73', 'p74', 'p75', 'p76', 'p77', 'p78', 'p79', 'p80', 'p81', 'p82', 'p83', 'p84', 'p85', 'p86', 'p87', 'p88', 'p89', 'p90', 'p91', 'p92', 'p93', 'p94', 'p95', 'p96']]

n_tasks = 96
y_train = np.ravel(y_train)
# 调整y_train的形状为(n_samples, n_tasks)
y_train = y_train.reshape(-1, n_tasks)

y_train.shape

(4337, 96)

In [39]:
df = pd.DataFrame(power_test)

# 创建新的DataFrame,将p1到p96作为行索引
new_data = []
start_date = datetime(2023, 5, 1, 0, 15)
for i in range(df.shape[0]):
    row = df.iloc[i]
    for j in range(1, 97):
        time = start_date + timedelta(minutes=15*(j-1))
        new_data.append([row['光伏用户编号'], row['综合倍率'], time])

power_test = pd.DataFrame(new_data, columns=['光伏用户编号', '综合倍率', '时间'])
power_test

,光伏用户编号,综合倍率,时间
0,f1,80,2023-05-01 00:15:00
1,f1,80,2023-05-01 00:30:00
2,f1,80,2023-05-01 00:45:00
3,f1,80,2023-05-01 01:00:00
4,f1,80,2023-05-01 01:15:00
...,...,...,...
79195,f9,8000,2023-05-01 23:00:00
79196,f9,8000,2023-05-01 23:15:00
79197,f9,8000,2023-05-01 23:30:00
79198,f9,8000,2023-05-01 23:45:00


In [40]:
# Convert '时间' column to datetime64 data type
weather_test['时间'] = pd.to_datetime(weather_test['时间'])
power_test['时间'] = pd.to_datetime(power_test['时间'])

# Now merge the DataFrames
test_data = pd.merge(weather_test,power_test, on=['光伏用户编号','时间'])
test_data = pd.merge(test_data, user_info_test, on=['光伏用户编号'])


# 特征工程
test_data['时间'] = pd.to_datetime(test_data['时间'])
test_data['月份'] = test_data['时间'].dt.month
test_data['日期'] = test_data['时间'].dt.day
test_data['小时'] = test_data['时间'].dt.hour
test_data['分钟'] = test_data['时间'].dt.minute

# 获取数值型列
numeric_cols = test_data.select_dtypes(include=['number']).columns
feature_cols = [col for col in numeric_cols if col not in ['p1', 'p2', ..., 'p96']]
X_test = test_data[feature_cols].values

In [41]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
# 对 f1-f9 进行独热编码
encoder = OneHotEncoder(sparse=False)
feature1_values = test_data['光伏用户编号'].values.reshape(-1, 1)
feature1_encoded = encoder.fit_transform(feature1_values)
feature1_encoded

# 选择特征和目标变量
X_test = test_data.drop('光伏用户编号', axis=1)
X_test = X_test.drop('时间', axis=1)
X_test = X_test.drop('光伏用户名称', axis=1)
X_test = np.column_stack((X_test, feature1_encoded))

# 标准化或归一化特征
scaler = MinMaxScaler()
X_test = scaler.fit_transform(X_test)
X_test = np.ravel(X_test)
X_test = X_test.reshape(-1, n_tasks, n_features)

X_test.shape

D:\anaconda1\envs\pm3env\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(825, 96, 27)

In [42]:
# 添加周期性特征
period_length = 24
# 将 X_train 展平为二维
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_train_new = np.column_stack([X_train_flat,
                               np.sin(2 * np.pi * np.arange(X_train.shape[0]) / period_length),
                               np.cos(2 * np.pi * np.arange(X_train.shape[0]) / period_length)])
X_test_flat = X_test.reshape(X_test.shape[0], -1)
X_test = np.column_stack([X_test_flat,
                               np.sin(2 * np.pi * np.arange(X_test.shape[0]) / period_length),
                               np.cos(2 * np.pi * np.arange(X_test.shape[0]) / period_length)])

In [72]:
import numpy as np

# 将 X_train 展平为二维
X_train_flat = X_train.reshape(X_train.shape[0], -1)

# 创建 GPR 模型
gpr = GaussianProcessRegressor(kernel=rbf_kernel, n_restarts_optimizer=10, alpha=1e-6)
print("training...)
# 训练模型
gpr.fit(X_train_new, y_train)

training...


GaussianProcessRegressor(alpha=1e-06, kernel=1**2 * RBF(length_scale=10),
                         n_restarts_optimizer=10)

In [20]:
from joblib import dump, load
# 保存模型
# dump(gpr, 'gpr_model.joblib')

# 加载模型
gpr = load('gpr_model.joblib')

In [43]:
from sklearn.metrics import mean_squared_error, r2_score
y_train_pred = gpr.predict(X_train_new.reshape(X_train.shape[0], -1))
X_train = np.nan_to_num(X_train, 0)
y_train = np.nan_to_num(y_train, 0)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
print(f"Train MSE: {train_mse:.4f}")
print(f"Train R^2: {train_r2:.4f}")

Train MSE: 0.0026
Train R^2: 0.9875


In [44]:
# 在测试集上预测
print("Predicting on test set...")
y_pred = gpr.predict(X_test.reshape(X_test.shape[0], -1)).ravel()

Predicting on test set...


In [45]:
power_test = pd.read_csv("data\A榜-测试集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')
power_test

,光伏用户编号,综合倍率,时间
0,f1,80,2023-5-1 0:00
1,f1,80,2023-5-2 0:00
2,f1,80,2023-5-3 0:00
3,f1,80,2023-5-4 0:00
4,f1,80,2023-5-5 0:00
...,...,...,...
820,f9,8000,2023-7-27 0:00
821,f9,8000,2023-7-30 0:00
822,f9,8000,2023-7-31 0:00
823,f9,8000,2023-5-15 0:00


In [46]:
# 重构预测结果
y_pred = y_pred.reshape(-1,96)
y_pred_df = pd.DataFrame(y_pred, columns=['p'+str(i) for i in range(1,97)])

# 在 power_test 中创建 p1 到 p96 列
for i in range(1, 97):
    power_test[f'p{i}'] = 0

# 将 y_pred_df 中的值拼接到 power_test 中对应的列
for col in y_pred_df.columns:
    power_test[col] = y_pred_df[col].values
power_test

,光伏用户编号,综合倍率,时间,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24,p25,p26,p27,p28,p29,p30,p31,p32,p33,p34,p35,p36,p37,p38,p39,p40,p41,p42,p43,p44,p45,p46,p47,p48,p49,p50,p51,p52,p53,p54,p55,p56,p57,p58,p59,p60,p61,p62,p63,p64,p65,p66,p67,p68,p69,p70,p71,p72,p73,p74,p75,p76,p77,p78,p79,p80,p81,p82,p83,p84,p85,p86,p87,p88,p89,p90,p91,p92,p93,p94,p95,p96
0,f1,80,2023-5-1 0:00,-0.000212,-0.000201,-0.000185,-0.000195,-0.000194,-0.000203,-0.000210,-0.000198,-0.000202,-0.000192,-0.000202,-0.000205,-0.000198,-0.000186,-0.000202,-0.000204,-0.000186,-0.000198,-0.000199,-0.000226,-0.000320,0.001441,0.005642,0.012811,0.018327,0.028555,0.039337,0.059652,0.079512,0.098292,0.127506,0.157245,0.194603,0.190781,0.212345,0.223686,0.252156,0.300660,0.271680,0.284961,0.310225,0.307171,0.290191,0.273756,0.288814,0.345502,0.344413,0.352432,0.358949,0.354386,0.335436,0.345496,0.313666,0.331513,0.311869,0.267585,0.244303,0.240845,0.181263,0.168093,0.153223,0.108393,0.094027,0.064705,0.058397,0.041900,0.034313,0.023858,0.017382,0.012717,0.008392,0.002809,0.000051,-0.002063,-0.001910,-0.001538,-0.001515,-0.000886,-0.000877,-0.000868,-0.000879,-0.000876,-0.000880,-0.000822,0.001432,-0.000808,-0.000816,-0.000456,-0.000410,-0.000357,-0.000261,-0.000256,-0.000252,-0.000231,-0.000239,-0.000238
1,f1,80,2023-5-2 0:00,-0.000216,-0.000208,-0.000195,-0.000207,-0.000203,-0.000214,-0.000217,-0.000206,-0.000211,-0.000205,-0.000211,-0.000218,-0.000205,-0.000199,-0.000205,-0.000208,-0.000200,-0.000202,-0.000210,-0.000222,-0.000301,0.000606,0.003064,0.007680,0.011795,0.021030,0.030331,0.050446,0.067202,0.082927,0.110554,0.135578,0.161436,0.171512,0.190462,0.199522,0.226791,0.274131,0.256380,0.273347,0.300681,0.299086,0.286301,0.271858,0.275235,0.318899,0.323726,0.341885,0.326606,0.322116,0.307431,0.314993,0.295351,0.314733,0.292975,0.258765,0.224115,0.228881,0.169396,0.160423,0.145903,0.097468,0.079555,0.056864,0.047490,0.034374,0.026266,0.017779,0.013099,0.009260,0.005182,0.000153,-0.001460,-0.002317,-0.001602,-0.001199,-0.001118,-0.000782,-0.000755,-0.000745,-0.000746,-0.000746,-0.000749,-0.000703,0.002659,-0.000692,-0.000709,-0.000436,-0.000402,-0.000337,-0.000260,-0.000254,-0.000248,-0.000227,-0.000235,-0.000234
2,f1,80,2023-5-3 0:00,-0.000195,-0.000187,-0.000178,-0.000191,-0.000184,-0.000194,-0.000196,-0.000186,-0.000193,-0.000189,-0.000191,-0.000200,-0.000188,-0.000184,-0.000184,-0.000188,-0.000184,-0.000184,-0.000194,-0.000198,-0.000242,0.000306,0.002042,0.005310,0.008734,0.016748,0.024879,0.042534,0.057241,0.070393,0.093967,0.113545,0.130243,0.147944,0.165931,0.171019,0.195841,0.236084,0.224646,0.240338,0.265730,0.264925,0.254425,0.242270,0.241391,0.271535,0.275902,0.295183,0.276257,0.272385,0.259663,0.266253,0.250822,0.265193,0.248428,0.220819,0.186038,0.192766,0.143386,0.136667,0.123694,0.080749,0.064339,0.047659,0.039364,0.028843,0.021023,0.013875,0.010227,0.007489,0.003857,-0.000414,-0.001697,-0.002100,-0.001298,-0.000934,-0.000834,-0.000662,-0.000633,-0.000623,-0.000618,-0.000619,-0.000622,-0.000584,0.002740,-0.000579,-0.000597,-0.000364,-0.000339,-0.000269,-0.000223,-0.000217,-0.000213,-0.000196,-0.000204,-0.000203
3,f1,80,2023-5-4 0:00,-0.000204,-0.000193,-0.000175,-0.000183,-0.000185,-0.000190,-0.000202,-0.000190,-0.000191,-0.000181,-0.000193,-0.000192,-0.000188,-0.000174,-0.000195,-0.000197,-0.000172,-0.000191,-0.000186,-0.000225,-0.000332,0.002014,0.007343,0.015752,0.021441,0.031152,0.042436,0.060661,0.082320,0.101885,0.128889,0.159085,0.202290,0.187714,0.212452,0.221944,0.250963,0.291596,0.259720,0.267006,0.291175,0.284350,0.264747,0.247559,0.274973,0.331334,0.320417,0.317753,0.343468,0.340855,0.316773,0.329762,0.287791,0.300135,0.285749,0.235280,0.225967,0.217905,0.164408,0.148391,0.138695,0.103189,0.095570,0.064486,0.062273,0.044086,0.037676,0.026185,0.018950,0.014423,0.010369,0.004828,0.001388,-0.001637,-0.001975,-0.001679,-0.001724,-0.000927,-0.000934,-0.000924,-0.000943,-0.000935,-0.000938,-0.000876,0.000204,

In [47]:

power_test = power_test.round(4)

# 将结果写入CSV文件,编码格式为UTF-8
power_test.to_csv('xxx.csv', encoding='utf-8', index=False)

In [48]:
pd.read_csv("xxx.csv",encoding='utf-8')

,光伏用户编号,综合倍率,时间,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24,p25,p26,p27,p28,p29,p30,p31,p32,p33,p34,p35,p36,p37,p38,p39,p40,p41,p42,p43,p44,p45,p46,p47,p48,p49,p50,p51,p52,p53,p54,p55,p56,p57,p58,p59,p60,p61,p62,p63,p64,p65,p66,p67,p68,p69,p70,p71,p72,p73,p74,p75,p76,p77,p78,p79,p80,p81,p82,p83,p84,p85,p86,p87,p88,p89,p90,p91,p92,p93,p94,p95,p96
0,f1,80,2023-5-1 0:00,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0003,0.0014,0.0056,0.0128,0.0183,0.0286,0.0393,0.0597,0.0795,0.0983,0.1275,0.1572,0.1946,0.1908,0.2123,0.2237,0.2522,0.3007,0.2717,0.2850,0.3102,0.3072,0.2902,0.2738,0.2888,0.3455,0.3444,0.3524,0.3589,0.3544,0.3354,0.3455,0.3137,0.3315,0.3119,0.2676,0.2443,0.2408,0.1813,0.1681,0.1532,0.1084,0.0940,0.0647,0.0584,0.0419,0.0343,0.0239,0.0174,0.0127,0.0084,0.0028,0.0001,-0.0021,-0.0019,-0.0015,-0.0015,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0008,0.0014,-0.0008,-0.0008,-0.0005,-0.0004,-0.0004,-0.0003,-0.0003,-0.0003,-0.0002,-0.0002,-0.0002
1,f1,80,2023-5-2 0:00,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0003,0.0006,0.0031,0.0077,0.0118,0.0210,0.0303,0.0504,0.0672,0.0829,0.1106,0.1356,0.1614,0.1715,0.1905,0.1995,0.2268,0.2741,0.2564,0.2733,0.3007,0.2991,0.2863,0.2719,0.2752,0.3189,0.3237,0.3419,0.3266,0.3221,0.3074,0.3150,0.2954,0.3147,0.2930,0.2588,0.2241,0.2289,0.1694,0.1604,0.1459,0.0975,0.0796,0.0569,0.0475,0.0344,0.0263,0.0178,0.0131,0.0093,0.0052,0.0002,-0.0015,-0.0023,-0.0016,-0.0012,-0.0011,-0.0008,-0.0008,-0.0007,-0.0007,-0.0007,-0.0007,-0.0007,0.0027,-0.0007,-0.0007,-0.0004,-0.0004,-0.0003,-0.0003,-0.0003,-0.0002,-0.0002,-0.0002,-0.0002
2,f1,80,2023-5-3 0:00,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,0.0003,0.0020,0.0053,0.0087,0.0167,0.0249,0.0425,0.0572,0.0704,0.0940,0.1135,0.1302,0.1479,0.1659,0.1710,0.1958,0.2361,0.2246,0.2403,0.2657,0.2649,0.2544,0.2423,0.2414,0.2715,0.2759,0.2952,0.2763,0.2724,0.2597,0.2663,0.2508,0.2652,0.2484,0.2208,0.1860,0.1928,0.1434,0.1367,0.1237,0.0807,0.0643,0.0477,0.0394,0.0288,0.0210,0.0139,0.0102,0.0075,0.0039,-0.0004,-0.0017,-0.0021,-0.0013,-0.0009,-0.0008,-0.0007,-0.0006,-0.0006,-0.0006,-0.0006,-0.0006,-0.0006,0.0027,-0.0006,-0.0006,-0.0004,-0.0003,-0.0003,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002
3,f1,80,2023-5-4 0:00,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0003,0.0020,0.0073,0.0158,0.0214,0.0312,0.0424,0.0607,0.0823,0.1019,0.1289,0.1591,0.2023,0.1877,0.2125,0.2219,0.2510,0.2916,0.2597,0.2670,0.2912,0.2843,0.2647,0.2476,0.2750,0.3313,0.3204,0.3178,0.3435,0.3409,0.3168,0.3298,0.2878,0.3001,0.2857,0.2353,0.2260,0.2179,0.1644,0.1484,0.1387,0.1032,0.0956,0.0645,0.0623,0.0441,0.0377,0.0262,0.0189,0.0144,0.0104,0.0048,0.0014,-0.0016,-0.0020,-0.0017,-0.0017,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,0.0002,-0.0009,-0.0009,-0.0004,-0.0004,-0.0003,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002
4,f1,80,2023-5-5 0:00,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0002,-0.0003,0.0013,0.0050,0.0114,0.0159,0.0249,0.0345,0.0540,0.0725,0.0897,0.1173,0.1462,0.1815,0.1802,0.2020,0.2110,0.2394,0.2869,0.2601,0.2745,0.3013,0.2945,0.2794,0.2654,0.2789,0.3330,0.3309,0.3408,0.3419,0.3371,0.3195,0.3306,0.3054,0.3209,0.3015,0.2593,0.2354,0.2347,0.1752,0.1625,0.1495,0.1057,0.0913,0.0637,0.0558,0.0396,0.0322,0.0218,0.0155,0.0111,0.0072,0.0020,-0.0001,-0.0020,-0.0018,-0.0015,-0.0015,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0009,-0.0008,0.0015,-0.0008,